In [0]:
--2. Compléter la couche Silver:
--Historisez dans le lakehouse en silver
--toutes les tables importées
--précédemment dans la couche bronze.


SHOW TABLES IN barbara_lakehouse.bronze;

DECLARE OR REPLACE load_date = current_timestamp();
VALUES load_date;

-- Product
CREATE TABLE IF NOT EXISTS barbara_lakehouse.silver.product (
  _tf_id                    BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id                INT,
  name                      STRING,
  product_number            STRING,
  color                     STRING,
  standard_cost             DECIMAL(19,4),
  list_price                DECIMAL(19,4),
  size                      STRING,
  weight                    DECIMAL(8,2),
  product_category_id       INT,
  product_model_id          INT,
  sell_start_date           TIMESTAMP,
  sell_end_date             TIMESTAMP,
  discontinued_date         TIMESTAMP,
  thumbnail_photo           BINARY,
  thumbnail_photo_filename  STRING,
  rowguid                   STRING,
  modified_date             TIMESTAMP,
  _tf_valid_from            TIMESTAMP,
  _tf_valid_to              TIMESTAMP
)
USING DELTA
;

MERGE INTO barbara_lakehouse.silver.product AS tgt
USING (
  SELECT
    ProductID              AS product_id,
    Name                   AS name,
    ProductNumber          AS product_number,
    Color                  AS color,
    StandardCost           AS standard_cost,
    ListPrice              AS list_price,
    Size                   AS size,
    Weight                 AS weight,
    ProductCategoryID      AS product_category_id,
    ProductModelID         AS product_model_id,
    SellStartDate          AS sell_start_date,
    SellEndDate            AS sell_end_date,
    DiscontinuedDate       AS discontinued_date,
    ThumbNailPhoto         AS thumbnail_photo,
    ThumbnailPhotoFileName AS thumbnail_photo_filename,
    rowguid                AS rowguid,
    ModifiedDate           AS modified_date
  FROM bronze.product
) AS src
ON tgt.product_id = src.product_id
AND tgt._tf_valid_to IS NULL

WHEN MATCHED AND (
       tgt.name                    IS DISTINCT FROM src.name
    OR tgt.product_number          IS DISTINCT FROM src.product_number
    OR tgt.color                   IS DISTINCT FROM src.color
    OR tgt.standard_cost           IS DISTINCT FROM src.standard_cost
    OR tgt.list_price              IS DISTINCT FROM src.list_price
    OR tgt.size                    IS DISTINCT FROM src.size
    OR tgt.weight                  IS DISTINCT FROM src.weight
    OR tgt.product_category_id     IS DISTINCT FROM src.product_category_id
    OR tgt.product_model_id        IS DISTINCT FROM src.product_model_id
    OR tgt.sell_start_date         IS DISTINCT FROM src.sell_start_date
    OR tgt.sell_end_date           IS DISTINCT FROM src.sell_end_date
    OR tgt.discontinued_date       IS DISTINCT FROM src.discontinued_date
    OR tgt.thumbnail_photo         IS DISTINCT FROM src.thumbnail_photo
    OR tgt.thumbnail_photo_filename IS DISTINCT FROM src.thumbnail_photo_filename
    OR tgt.rowguid                 IS DISTINCT FROM src.rowguid
    OR tgt.modified_date           IS DISTINCT FROM src.modified_date
)
THEN
  -- 1) Close de l’ancienne version (update SCD2)
  UPDATE SET
    tgt._tf_valid_to = load_date

WHEN NOT MATCHED BY SOURCE
AND tgt._tf_valid_to IS NULL
THEN
  -- 2) Close des lignes supprimées dans la source
  UPDATE SET
    tgt._tf_valid_to = load_date
;

MERGE INTO silver.product AS tgt
USING (
  SELECT
    ProductID              AS product_id,
    Name                   AS name,
    ProductNumber          AS product_number,
    Color                  AS color,
    StandardCost           AS standard_cost,
    ListPrice              AS list_price,
    Size                   AS size,
    Weight                 AS weight,
    ProductCategoryID      AS product_category_id,
    ProductModelID         AS product_model_id,
    SellStartDate          AS sell_start_date,
    SellEndDate            AS sell_end_date,
    DiscontinuedDate       AS discontinued_date,
    ThumbNailPhoto         AS thumbnail_photo,
    ThumbnailPhotoFileName AS thumbnail_photo_filename,
    rowguid                AS rowguid,
    ModifiedDate           AS modified_date
  FROM bronze.product
) AS src
ON tgt.product_id = src.product_id
AND tgt._tf_valid_to IS NULL

WHEN NOT MATCHED THEN
  INSERT (
    product_id,
    name,
    product_number,
    color,
    standard_cost,
    list_price,
    size,
    weight,
    product_category_id,
    product_model_id,
    sell_start_date,
    sell_end_date,
    discontinued_date,
    thumbnail_photo,
    thumbnail_photo_filename,
    rowguid,
    modified_date,
    _tf_valid_from,
    _tf_valid_to
  )
  VALUES (
    src.product_id,
    src.name,
    src.product_number,
    src.color,
    src.standard_cost,
    src.list_price,
    src.size,
    src.weight,
    src.product_category_id,
    src.product_model_id,
    src.sell_start_date,
    src.sell_end_date,
    src.discontinued_date,
    src.thumbnail_photo,
    src.thumbnail_photo_filename,
    src.rowguid,
    src.modified_date,
    load_date,   -- _tf_valid_from
    NULL         -- _tf_valid_to
  )
;


-- ProductCategory

CREATE TABLE IF NOT EXISTS barbara_lakehouse.silver.productcategory (
  _tf_id                     BIGINT GENERATED ALWAYS AS IDENTITY,
  product_category_id         INT,
  parent_product_category_id  INT,
  name                        STRING,
  rowguid                     STRING,      -- (photo: char(36))
  modified_date               TIMESTAMP,
  _tf_valid_from              TIMESTAMP,
  _tf_valid_to                TIMESTAMP,
  _tf_create_date             TIMESTAMP,
  _tf_update_date             TIMESTAMP
)
USING DELTA
;

MERGE INTO barbara_lakehouse.silver.productcategory AS tgt
USING (
  SELECT
    ProductCategoryID       AS product_category_id,
    ParentProductCategoryID AS parent_product_category_id,
    Name                    AS name,
    rowguid                 AS rowguid,
    ModifiedDate            AS modified_date
  FROM barbara_lakehouse.bronze.productcategory
) AS src
ON tgt.product_category_id = src.product_category_id
AND tgt._tf_valid_to IS NULL

WHEN MATCHED AND (
       tgt.parent_product_category_id IS DISTINCT FROM src.parent_product_category_id
    OR tgt.name                       IS DISTINCT FROM src.name
    OR tgt.rowguid                    IS DISTINCT FROM src.rowguid
    OR tgt.modified_date              IS DISTINCT FROM src.modified_date
)
THEN
  -- 1) Close ancienne version
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED BY SOURCE
AND tgt._tf_valid_to IS NULL
THEN
  -- 2) Close des "deleted"
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date
;

MERGE INTO barbara_lakehouse.silver.productcategory AS tgt
USING (
  SELECT
    ProductCategoryID       AS product_category_id,
    ParentProductCategoryID AS parent_product_category_id,
    Name                    AS name,
    rowguid                 AS rowguid,
    ModifiedDate            AS modified_date
  FROM barbara_lakehouse.bronze.productcategory
) AS src
ON tgt.product_category_id = src.product_category_id
AND tgt._tf_valid_to IS NULL

WHEN NOT MATCHED THEN
  INSERT (
    product_category_id,
    parent_product_category_id,
    name,
    rowguid,
    modified_date,
    _tf_valid_from,
    _tf_valid_to,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.product_category_id,
    src.parent_product_category_id,
    src.name,
    src.rowguid,
    src.modified_date,
    load_date,  -- _tf_valid_from
    NULL,       -- _tf_valid_to
    load_date,  -- _tf_create_date
    load_date   -- _tf_update_date
  )
;

-- ProductModel

CREATE TABLE IF NOT EXISTS barbara_lakehouse.silver.productmodel (
  _tf_id            BIGINT GENERATED ALWAYS AS IDENTITY,

  product_model_id  INT,
  name              STRING,
  catalog_description STRING,
  rowguid           STRING,
  modified_date     TIMESTAMP,

  _tf_valid_from    TIMESTAMP,
  _tf_valid_to      TIMESTAMP,
  _tf_create_date   TIMESTAMP,
  _tf_update_date   TIMESTAMP
)
USING DELTA;

MERGE INTO barbara_lakehouse.silver.productmodel AS tgt
USING (
  SELECT
    ProductModelID        AS product_model_id,
    Name                  AS name,
    CatalogDescription    AS catalog_description,
    rowguid               AS rowguid,
    ModifiedDate          AS modified_date
  FROM barbara_lakehouse.bronze.productmodel
) AS src
ON tgt.product_model_id = src.product_model_id
AND tgt._tf_valid_to IS NULL

WHEN MATCHED AND (
       tgt.name                IS DISTINCT FROM src.name
    OR tgt.catalog_description IS DISTINCT FROM src.catalog_description
    OR tgt.rowguid             IS DISTINCT FROM src.rowguid
    OR tgt.modified_date       IS DISTINCT FROM src.modified_date
) THEN
  -- Close ancienne version
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED BY SOURCE
AND tgt._tf_valid_to IS NULL THEN
  -- Close des lignes supprimées en source
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date
;

MERGE INTO barbara_lakehouse.silver.productmodel AS tgt
USING (
  SELECT
    ProductModelID        AS product_model_id,
    Name                  AS name,
    CatalogDescription    AS catalog_description,
    rowguid               AS rowguid,
    ModifiedDate          AS modified_date
  FROM barbara_lakehouse.bronze.productmodel
) AS src
ON tgt.product_model_id = src.product_model_id
AND tgt._tf_valid_to IS NULL

WHEN NOT MATCHED THEN
  INSERT (
    product_model_id,
    name,
    catalog_description,
    rowguid,
    modified_date,
    _tf_valid_from,
    _tf_valid_to,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.product_model_id,
    src.name,
    src.catalog_description,
    src.rowguid,
    src.modified_date,
    load_date,
    NULL,
    load_date,
    load_date
  )
;